In [ ]:
# scraped these from google maps on parker's resturant

In [10]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

df = pd.read_csv("Parkers_review.csv")
df.head()

,text,label,topic
0,We had a wonderful dining experience at Parker...,positive,Service
1,A lovely place to relax after shopping! 🛍️✨ My...,positive,Service
2,"The ingredients at Parker’s seem decent, but t...",negative,Price
3,We had an incredible breakfast at Parker’s thi...,positive,Food
4,We went for the Matilda cske and it fid not di...,positive,Food


In [11]:
df["label"].value_counts()

,count
label,
positive,59
negative,9


## Sentiment 3 models

In [12]:
m1 = pipeline("text-classification", model="cardiffnlp/twitter-xlm-roberta-base-sentiment")
m2 = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")
m3 = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [13]:
def clean_label(model_name, raw_label):
    raw_label = raw_label.lower()
    if model_name == "m3":
        stars = int(raw_label[0])
        return "positive" if stars >= 4 else "negative"
    if "pos" in raw_label:
        return "positive"
    return "negative"


def run_model(model, model_name, text):
    out = model(text[:512])[0]
    return {
        "label": out["label"],
        "score": out["score"],
        "clean_label": clean_label(model_name, out["label"]),
        "metadata": "huggingface_AI_model",
    }


run_model(m1, "m1", df["text"][0])

{'label': 'positive',
 'score': 0.883211612701416,
 'clean_label': 'positive',
 'metadata': 'huggingface_AI_model'}

In [14]:
# quick test on a few
for t in df["text"][:3]:
    print(t[:60], "...")
    print(" m1:", run_model(m1, "m1", t)["clean_label"])
    print(" m2:", run_model(m2, "m2", t)["clean_label"])
    print(" m3:", run_model(m3, "m3", t)["clean_label"])
    print()

We had a wonderful dining experience at Parker’s. The food w ...
 m1: positive
 m2: positive
 m3: positive

A lovely place to relax after shopping! 🛍️✨ My mum and I sto ...
 m1: positive
 m2: positive
 m3: positive

The ingredients at Parker’s seem decent, but the whole exper ...
 m1: positive
 m2: negative
 m3: negative



In [15]:
df["pred_m1"] = df["text"].apply(lambda t: run_model(m1, "m1", t)["clean_label"])
df["pred_m2"] = df["text"].apply(lambda t: run_model(m2, "m2", t)["clean_label"])
df["pred_m3"] = df["text"].apply(lambda t: run_model(m3, "m3", t)["clean_label"])
df.head()

,text,label,topic,pred_m1,pred_m2,pred_m3
0,We had a wonderful dining experience at Parker...,positive,Service,positive,positive,positive
1,A lovely place to relax after shopping! 🛍️✨ My...,positive,Service,positive,positive,positive
2,"The ingredients at Parker’s seem decent, but t...",negative,Price,positive,negative,negative
3,We had an incredible breakfast at Parker’s thi...,positive,Food,positive,positive,positive
4,We went for the Matilda cske and it fid not di...,positive,Food,positive,positive,positive


In [16]:
for col in ["pred_m1", "pred_m2", "pred_m3"]:
    acc = accuracy_score(df["label"], df[col])
    f1 = f1_score(df["label"], df[col], pos_label="negative")  # negative is the minority class here
    print(col, "accuracy:", round(acc, 2), "f1 (negative class):", round(f1, 2))

pred_m1 accuracy: 0.94 f1 (negative class): 0.71
pred_m2 accuracy: 0.96 f1 (negative class): 0.8
pred_m3 accuracy: 0.97 f1 (negative class): 0.89


### Wrong outputs



In [17]:
wrong = df[(df["label"] != df["pred_m1"]) | (df["label"] != df["pred_m2"]) | (df["label"] != df["pred_m3"])]
wrong[["text", "label", "pred_m1", "pred_m2", "pred_m3"]]
#a few of these are reviews that complain about price or one dish but still end on a positive note about the staff.

,text,label,pred_m1,pred_m2,pred_m3
2,"The ingredients at Parker’s seem decent, but t...",negative,positive,negative,negative
9,Nice food but I'd say a bit overpriced. Paul w...,negative,positive,positive,positive
16,This was a pleasant find in the City Centre Ma...,negative,positive,negative,negative
25,I had a pleasant experience at Parker's. The s...,negative,positive,positive,negative
43,24.7.2026 All was good. Food taste good. It is...,positive,positive,positive,negative
44,way over rated and over priced. We had Greek s...,negative,negative,positive,negative


In [18]:
# m1 looks to be the best

## zero shot

In [19]:
df["topic"].value_counts()

,count
topic,
Service,44
Food,14
Price,6
Atmosphere,3
Cleanliness,1


In [20]:
candidate_labels = ["Food", "Service", "Price", "Cleanliness", "Atmosphere"]
zs = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

def run_zs(text):
    out = zs(text[:512], candidate_labels)
    return {
        "label": out["labels"][0],
        "score": out["scores"][0],
        "metadata": "huggingface_AI_model",
    }

df["pred_topic"] = df["text"].apply(lambda t: run_zs(t)["label"])
acc = accuracy_score(df["topic"], df["pred_topic"])
print("topic accuracy:", round(acc, 2))

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

topic accuracy: 0.59


In [21]:
df[df["topic"] != df["pred_topic"]][["text", "topic", "pred_topic"]]
# most reviews mention both food and service, so the zero-shot model sometimes picks "Food" when I tagged it "Service" or vice versa, not really wrong though.

,text,topic,pred_topic
2,"The ingredients at Parker’s seem decent, but t...",Price,Food
3,We had an incredible breakfast at Parker’s thi...,Food,Atmosphere
7,We were lucky that we skipped all other restau...,Food,Service
9,Nice food but I'd say a bit overpriced. Paul w...,Price,Food
11,I had a wonderful dining experience at Parker’...,Service,Atmosphere
14,We tried Parker's for lunch and the experience...,Service,Food
15,"Hade a wonderful time at Parker’s restaurant, ...",Service,Food
16,This was a pleasant find in the City Centre Ma...,Food,Service
20,"We came here for dinner, and it was really goo...",Service,Food
22,the place was amazing the foid was unreal real...,Service,Food
